# Missing Indicator — Flagging *That* a Value Was Missing

Imputing fills in a plausible value, but it also **erases the fact that it was missing** — which can itself be useful information (e.g. people who skip an income question may differ systematically from those who don't). A **missing indicator** column (1 if the value was missing, 0 otherwise) lets a model use both: the filled-in value *and* the fact that it needed filling.

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import MissingIndicator,SimpleImputer

In [2]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.head()

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
X_train.head()

## 2. Baseline — Plain Mean Imputation, No Indicator

In [ ]:
si = SimpleImputer()
X_train_trf = si.fit_transform(X_train)
X_test_trf = si.transform(X_test)

In [ ]:
X_train_trf

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

clf.fit(X_train_trf,y_train)

y_pred = clf.predict(X_test_trf)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

## 3. Adding a Missing Indicator Manually — `MissingIndicator`

`MissingIndicator` doesn't impute anything itself — it only outputs a 0/1 flag per value, for columns that actually had missing data (`mi.features_` shows which column indices qualified).

In [ ]:
mi = MissingIndicator()

mi.fit(X_train)

In [ ]:
mi.features_

In [ ]:
X_train_missing = mi.transform(X_train)

In [ ]:
X_train_missing

In [ ]:
X_test_missing = mi.transform(X_test)

In [ ]:
X_test_missing

Attaching the flag as a new column on both train and test:

In [ ]:
X_train['Age_NA'] = X_train_missing

In [ ]:
X_test

In [ ]:
X_test['Age_NA'] = X_test_missing

In [ ]:
X_train

### Now Impute the Original Columns as Usual

The indicator column is a flag, not something to impute — the original `Age`/`Fare` columns still need their own missing values filled separately, same as the baseline above, just now alongside the new `Age_NA` column.

In [ ]:
si = SimpleImputer()

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()

clf.fit(X_train_trf2,y_train)

y_pred = clf.predict(X_test_trf2)

from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

## 4. The Shortcut — `SimpleImputer(add_indicator=True)`

Doing this by hand (as above) works, but `SimpleImputer` can add the missing-indicator column **automatically**, in the same step as imputing — one object instead of two.

In [ ]:
si = SimpleImputer(add_indicator=True)

In [ ]:
X_train = si.fit_transform(X_train)

In [ ]:
X_test = si.transform(X_test)

### ⚠️ Bug: Evaluating Stale Variables

This cell reuses `X_train_trf2` / `X_test_trf2` — the variables from the **earlier, plain** `SimpleImputer` (Section 2), with no indicator column at all. The newly-transformed `X_train` / `X_test` from the two cells above (with `add_indicator=True`) were never actually evaluated. Fixed below to use the right variables.

In [ ]:
clf = LogisticRegression()
clf.fit(X_train, y_train)   # X_train here is the add_indicator=True version from the cells above
y_pred = clf.predict(X_test)
accuracy_score(y_test, y_pred)


## Summary

| Task | Method |
|---|---|
| Get a 0/1 flag for missingness, standalone | `MissingIndicator()` |
| See which columns had any missing values | `mi.features_` |
| Impute **and** add the flag in one step | `SimpleImputer(add_indicator=True)` |

### When this actually helps
If missingness itself carries signal (survey non-response correlating with the target, say), the indicator gives a model access to that signal; if missingness really is random noise, the extra column costs little and rarely hurts. Worth comparing accuracy with vs. without it, as this notebook does, rather than assuming either way.